In [4]:
import duckdb
import os

# -----------------------------
# CONNECT TO DATABASE
# -----------------------------
conn = duckdb.connect(
    "../database/airport_ops.duckdb"
)

# -----------------------------
# CREATE GOLD TABLE
# -----------------------------
conn.execute("""

CREATE OR REPLACE TABLE
gold_airport_dashboard AS

SELECT

    iata_code,
    airport_name,
    municipality,
    iso_country,

    nearby_flights,

    temperature_c,
    wind_kph,
    visibility_km,
    weather_condition,

    weather_risk,
    airport_stress_score,

    CASE

        WHEN airport_stress_score >= 100
        THEN 'Critical'

        WHEN airport_stress_score >= 50
        THEN 'High'

        WHEN airport_stress_score >= 20
        THEN 'Medium'

        ELSE 'Low'

    END
    AS stress_level

FROM airport_metrics

""")

print("Gold table created!")

# -----------------------------
# PREVIEW DATA
# -----------------------------
preview = conn.sql("""
SELECT *
FROM gold_airport_dashboard
LIMIT 20
""").df()

print(preview)

# -----------------------------
# CREATE EXPORT FOLDER
# -----------------------------
os.makedirs(
    "../exports",
    exist_ok=True
)

# -----------------------------
# EXPORT PARQUET
# -----------------------------
gold = conn.sql("""
SELECT *
FROM gold_airport_dashboard
""").df()

gold.to_parquet(
    "../exports/gold_airport_dashboard.parquet",
    index=False
)

print(
    "Parquet exported successfully!"
)

# -----------------------------
# CLOSE CONNECTION
# -----------------------------
conn.close()

print("Done!")

Gold table created!
   iata_code                                 airport_name   municipality  \
0        RKV                   Reykjavík Domestic Airport      Reykjavík   
1        PRN  Priština Adem Jashari International Airport      Prishtina   
2        YAG               Fort Frances Municipal Airport   Fort Frances   
3        YAZ                  Tofino / Long Beach Airport         Tofino   
4        QBC                          Bella Coola Airport    Bella Coola   
5        YBK                           Baker Lake Airport     Baker Lake   
6        YCC                    Cornwall Regional Airport       Cornwall   
7        YEG               Edmonton International Airport       Edmonton   
8        YEN                              Estevan Airport        Estevan   
9        YTM         Mont-Tremblant International Airport      La Macaza   
10       YGP                 Michel-Pouliot Gaspé Airport          Gaspé   
11       YHZ    Halifax / Stanfield International Airport        Hal